<a href="https://colab.research.google.com/github/thomashooks53/undergrad_ml_assignments/blob/main/03_assignment_linear_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 3: Linear Models

**Q1.** Please answer the following questions in your own words.

1. What makes a model "linear"? "Linear" in what?

    - A model is “linear” if it is linear in its coefficients, not in the input variables. That means each coefficient multiplies a term and they are added together, with no coefficients multiplied by each other or inside nonlinear functions.

2. How do you interpret the coefficient for a dummy/one-hot-encoded variable? (Hint: how do you handle the intercept of the model?)

    - The coefficient represents the difference in the predicted outcome compared to the reference category, holding all other variables constant. The intercept corresponds to the baseline group, so the dummy variable coefficient shows how much you add or subtract from that baseline when the dummy equals 1.

3. Can linear regression be used for classification? Explain why, or why not.

    - Yes, but it doesn't work very well. Linear regression can produce predictions outside the range [0,1], which don’t make sense as probabilities. It also assumes constant variance and normally distributed errors, which isn't usuallt the case in classification problems. That’s why methods like logistic regression are preferred.

4. What are signs that your linear model is over-fitting?

    - Overfitting is indicated when the model performs very well on training data but poorly on test data. Other signs include very large coefficients, high sensitivity to small changes in data, and capturing noise rather than true patterns.

5. Clearly explain multi-colinearity using the two-stage least squares technique.

    - Multicollinearity occurs when one predictor can be closely explained by other predictors, making it hard to isolate its effect. In a two-stage idea:
      - First, regress the problematic variable on the others to see how predictable it is.
      - If it is highly predictable, then in the second stage, its unique variation (what’s left after removing overlap) is very small, so its coefficient becomes unstable and hard to estimate reliably.

6. How can you incorporate nonlinear relationships between your target/response/dependent/outcome variable $y$ and your features/control/response/independent variables $x$ into your analysis?

    - You can include transformed or engineered features such as polynomial terms (e.g., $x$<sup>2</sup>), interaction terms (e.g., $x$<sub>1</sub>$x$<sub>2</sub>), or functions like logs or exponentials. Even though the relationship in $x$ is nonlinear, the model remains linear in coefficients.

7. What is the interpretation of the slope coefficient in a linear regression?

    - The slope coefficient represents the expected change in the response variable $y$ for a one-unit increase in the predictor, holding all other variables constant.

8. Compare the train/test split and $k$-fold cross validation.

    - Train/test split: Simple and fast; data is split once into training and testing sets. However, results can depend heavily on how the split is done.
    - $k$-fold cross-validation: Data is split into $k$ parts, and the model is trained/tested multiple times. This gives a more reliable estimate of performance but is more computationally expensive.

9. How is the $k$ in $k$-fold cross validation typically selected?

    - Choosing $k$ is a trade-off between time and accuracy. If $k$ is too small, you aren't using enough data to get a good training signal. If $k$ is too large, the process takes too long and can lead to high variance. Most people use $k=5$ or $k=10$. These values provide a great balance, giving a stable estimate of the model's accuracy without requiring an excessive amount of computing power.


### **Run:** get_data.py

In [ ]:
import urllib.request
import os
import zipfile
import os

def download_data(force=False):
    """Download and extract course data from Zenodo."""

    zip_path = 'data.zip'
    data_dir = './data'

    if not os.path.exists(zip_path) or force:
        print("Downloading course data...")
        urllib.request.urlretrieve(
            'https://zenodo.org/records/18235955/files/data.zip?download=1',
            zip_path
        )
        print("Download complete")

    if not os.path.exists(data_dir) or force:
        print("Extracting data files...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
        print("Data extracted")

    return data_dir


if __name__ == "__main__":
    download_data()

**Q2.** Load `./data/Q1_clean.csv`. The data include

- `Price` per night
- `Review Scores Rating`: The average rating for the property
- `Neighbourhood`: The bourough of NYC. Note the space, or rename the variable.

1. Compute the average prices and scores by `Neighbourhood`; which bourough is the most expensive on average? Create a kernel density plot of price and log price, grouping by `Neighbourhood`.
2. Regress price on `Neighbourhood` by creating the appropriate dummy/one-hot-encoded variables (Are you dropping the first category, or the intercept of the regresssion?). Compare the coefficients in the regression to the table from part 1 (the answer depends on how you handled the dummy variable trap). How are the conditional group means and the estimated coefficients related?
3. Regress price on `Review Scores Rating` and a constant/intercept. Interpret the slope coefficient clearly in words.
4. Regress price on both `Neighbourhood` and `Review Scores Rating`. How does the slope coefficient on `Review Scores Rating` change? How do the neighborhood averages change?
5. Here is a puzzle: Regress price on a constant, and a separate slope coefficient for each neighborhood for `Review Scores Rating`. Are the slopes similar across neighborhoods, or not?
6. Use cross validation to evaluate the models from parts 4, 5, and 6.

**Q3.** This question is a case study for linear models. The data are about car prices. In particular, they include:

  - `Price`: In Indian rupees
  - `Seating_Capacity`: Number of seats
  - `Body_Type`: crossover, hatchback, muv, sedan, suv
  - `Make_Year`: The year the car was made

  1. Load `cars_hw.csv`. Summarize the `Price` variable and create a kernel density plot. Use `.groupby()` and `.describe()` to summarize prices by `Body_Type`. Make a grouped kernel density plot by `Body_Type`. Which car types are the most expensive? Which have the most variance?
  2. Regress `Price` on `Seating_Capacity`. What's the slope coefficient? Interpret it. Now treat `Seating_Capacity` as a one hot encoded variable, and regress price on it as if it is categorical. Are the differences in price roughly linear in the number of seats?
  3. Use `Make_Year` to create a new variable that corresponds to the age of the vehicle. Use 10-fold cross validation to determine the optimal number of powers of `Age` to include in a regression of `Price` on `Age`.
  4. Plot `Price` against `Age`, and then model-predicted price against `Age`. Does the model accurately fit the patterns in the data?


**Q4.** This question refers to the `heart_hw.csv` data. It contains three variables:

  - `y`: Whether the individual survived for three years, coded 0 for death and 1 for survival
  - `age`: Patient's age
  - `transplant`: labelled `control` for not receiving a transplant and labelled `treatment` for receiving a transplant

Since a heart transplant is a dangerous operation and even people who successfully get heart transplants might suffer later complications, we want to look at whether a group of transplant recipients tends to survive longer than a comparison group who does not get the procedure.

1. Compute (a) the proportion of people who survive in the control group and (b) in the treatment group. Compute (b) minus (a) as the average treatment effect (ATE). What is the ATE on three-year survival for heart transplant interventions?
2. Regress `y` on `transplant` using a linear model with a constant. Compare the intercept and transplant coefficient to the numbers you computed in part 1.
3. Regress `y` on transplant and age. How does the coefficient on transplant change when you control for age?What do the intercept and slope represent? Plot the predicted probability of survival by age, and add a dashed horizontal line for the ATE from part 1. For what ages is the ATE over- or under-estimating the impact of a transplant?
4. Now, include `transplant`, `age`, and `transplant * age` as controls. Repeat your analysis from part 3. How does allowing age and transplant to interact change your predictions? What pattern do you notice?
5. Use 10-fold cross validation to evaluate the predictive accuracy of these models.
6. Imagine this model is used to prioritize transplant access. What are your concerns about model construction and deployment?
